In [6]:
#Full code discretely monitored up and out call barrier option with antithetic variance reduction
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math 
import seaborn as sns
import scipy.stats as stats

np.random.seed(42)

#discretly monitored up and out call barrier option
# Model Parameters 
S0 = 100        # Start price
K = 100         # Strike
L = 120         # Barrier level
T = 1.0         # Duration (in years)
r = 0.05        # Risk-free interst rate
sigma = 0.4     # Volatility
d  = 0.03       # Dividend yield (continuous)

# Monte-Carlo-Parameter
n_sim = 100000     # Number of simulations
n_steps = 252       # Number of time discretisations (1 day)
dt = T / n_steps    # Timestep


#matrix with standard-normally distributed random numbers 
Z = np.random.normal(size=(n_sim, n_steps)) 
Z_anti = -Z  # antithetic counterpart

#structure for all the simulated price history
S = np.zeros((n_sim, n_steps)) 
S_anti = np.zeros((n_sim, n_steps))

#every first column of each simulation starts with value S_0
S[:, 0] = S0 
S_anti[:, 0] = S0

#GBM SDE with dividend yield
for t in range(1, n_steps):
    S[:, t] = S[:, t - 1] * np.exp((r - d - 0.5 * sigma**2) 
    * dt + sigma * np.sqrt(dt) * Z[:, t])
    S_anti[:, t] = S_anti[:, t - 1] * np.exp((r - d - 0.5 * sigma**2) 
    * dt + sigma * np.sqrt(dt) * Z_anti[:, t])

#hitting time 
knock = np.any(S >= L, axis=1)
knock_anti = np.any(S_anti >= L, axis=1)


#payoff  
payoff = np.where(knock, 0.0, np.maximum(S[:, -1] - K, 0.0)) #[V(T)]
payoff_anti = np.where(knock_anti, 0.0, np.maximum(S_anti[:, -1] - K, 0.0))

# Average both payoffs for variance reduction
payoff_avg = 0.5 * (payoff + payoff_anti)

#discounted price
discounted_price = np.exp(-r * T) * np.mean(payoff_avg)


#print
# Output
print (f"Discountet Price: {discounted_price:.4f}")


Discountet Price: 0.2543
